# Chapter 8.5: Model Registration in SGLang

## Learning Objectives

By the end of this notebook, you will:

1. **Understand** SGLang's model registry system
2. **Know** the model class interface requirements
3. **Master** the weight loading pipeline in SGLang
4. **Map** configurations from HuggingFace to SGLang models
5. **Compare** SGLang vs vLLM model registration
6. **Walk through** SGLang source code
7. **Register** a model in SGLang

---

## Prerequisites
- Chapter 8.1 (vLLM Model Registration)
- Familiarity with transformer architectures
- Basic understanding of SGLang's runtime

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part8/chapter_8.5_sglang_registration.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part8/chapter_8.5_sglang_registration.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. SGLang Architecture Overview

SGLang has a similar but distinct model registration system compared to vLLM.

```
SGLang Model System
===================

                    User: launch_server(model="meta-llama/Llama-3-8B")
                                     |
                                     v
                            Load HuggingFace config
                                     |
                                     v
                    architectures: ["LlamaForCausalLM"]
                                     |
                                     v
                         +---------------------+
                         | ModelRegistry        |
                         | (arch -> model_cls)  |
                         +---------------------+
                                     |
                                     v
                sglang.srt.models.llama.LlamaForCausalLM
                                     |
                                     v
                    Instantiate + Load Weights
                                     |
                                     v
                              Ready for inference
```

Key differences from vLLM:
1. SGLang uses `sglang.srt.models` package instead of `vllm.model_executor.models`
2. SGLang has its own attention backend (FlashInfer-based)
3. Forward method has `extend` vs `decode` modes
4. Weight loading is similar but uses SGLang's own utilities

## 2. The SGLang Model Registry

SGLang's model registry is defined in `sglang/srt/models/registry.py`.

In [ ]:
# SGLang model registry structure
# File: sglang/srt/models/registry.py

import torch
import torch.nn as nn
from typing import Dict, Type, Tuple, List, Optional

# The registry maps architecture names to (module_name, class_name)
# Similar to vLLM but in the sglang.srt.models namespace

_MODELS = {
    # Text-only models
    "LlamaForCausalLM": ("llama", "LlamaForCausalLM"),
    "MistralForCausalLM": ("llama", "LlamaForCausalLM"),  # Reuses Llama
    "Qwen2ForCausalLM": ("qwen2", "Qwen2ForCausalLM"),
    "Gemma2ForCausalLM": ("gemma2", "Gemma2ForCausalLM"),
    "DeepseekV2ForCausalLM": ("deepseek_v2", "DeepseekV2ForCausalLM"),
    "MixtralForCausalLM": ("mixtral", "MixtralForCausalLM"),
    "Phi3ForCausalLM": ("phi3", "Phi3ForCausalLM"),
    "GPT2LMHeadModel": ("gpt2", "GPT2LMHeadModel"),
    
    # Multimodal models
    "LlavaForConditionalGeneration": ("llava", "LlavaForConditionalGeneration"),
    "LlavaNextForConditionalGeneration": ("llava_next", "LlavaNextForConditionalGeneration"),
    "Qwen2VLForConditionalGeneration": ("qwen2_vl", "Qwen2VLForConditionalGeneration"),
    
    # Embedding models
    "LlamaModel": ("llama_embedding", "LlamaEmbeddingModel"),
}

print(f"SGLang supports {len(_MODELS)} architectures")
print("\nRegistered models:")
for arch, (module, cls) in sorted(_MODELS.items()):
    print(f"  {arch:45s} -> sglang.srt.models.{module}.{cls}")

In [ ]:
import importlib

class SGLangModelRegistry:
    """
    Simplified SGLang model registry.
    
    Actual file: sglang/srt/models/registry.py
    """
    
    _models: Dict[str, Tuple[str, str]] = {}
    _user_models: Dict[str, Type] = {}
    
    @classmethod
    def register(cls, arch_name: str, module_name: str, class_name: str):
        """Register a built-in model."""
        cls._models[arch_name] = (module_name, class_name)
    
    @classmethod
    def register_custom(cls, arch_name: str, model_class: Type):
        """Register a custom user model."""
        cls._user_models[arch_name] = model_class
    
    @classmethod
    def get_model_class(cls, architectures: List[str]) -> Type:
        """
        Resolve architecture name to model class.
        
        Priority: user models > built-in models
        """
        for arch in architectures:
            # Check user models first
            if arch in cls._user_models:
                return cls._user_models[arch]
            
            # Check built-in models
            if arch in cls._models:
                module_name, class_name = cls._models[arch]
                module = importlib.import_module(
                    f"sglang.srt.models.{module_name}"
                )
                return getattr(module, class_name)
        
        raise ValueError(
            f"Unsupported architecture: {architectures}. "
            f"Supported: {list(cls._models.keys())}"
        )
    
    @classmethod
    def list_models(cls) -> List[str]:
        return sorted(set(list(cls._models.keys()) + list(cls._user_models.keys())))

# Demo
registry = SGLangModelRegistry()
registry.register("LlamaForCausalLM", "llama", "LlamaForCausalLM")
registry.register("Qwen2ForCausalLM", "qwen2", "Qwen2ForCausalLM")

print(f"Registered: {registry.list_models()}")
print(f"\nResolution flow:")
print(f"  architectures=['LlamaForCausalLM'] -> sglang.srt.models.llama.LlamaForCausalLM")

## 3. SGLang Model Class Interface

SGLang models have a slightly different interface than vLLM models.

In [ ]:
from typing import Iterable, Set

class SGLangModelInterface(nn.Module):
    """
    Interface that SGLang models must implement.
    
    Key differences from vLLM:
    1. forward() receives ForwardBatch instead of separate tensors
    2. Uses FlashInfer attention backend
    3. Has explicit extend/decode handling in attention
    """
    
    def __init__(
        self,
        config,              # HuggingFace model config
        quant_config=None,   # Quantization config
    ):
        super().__init__()
    
    def forward(
        self,
        input_ids: torch.Tensor,       # [num_tokens]
        positions: torch.Tensor,        # [num_tokens]
        forward_batch,                  # ForwardBatch object
        input_embeds: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Forward pass.
        
        The ForwardBatch contains:
        - Attention metadata (prefill/decode info)
        - KV cache references
        - Batch information
        - Sampling parameters
        
        Returns: logits [num_tokens, vocab_size]
        """
        raise NotImplementedError
    
    def load_weights(
        self,
        weights: Iterable[Tuple[str, torch.Tensor]],
    ):
        """Load weights from HuggingFace checkpoint."""
        raise NotImplementedError

print("SGLang Model Interface:")
print("  __init__(config, quant_config)")
print("  forward(input_ids, positions, forward_batch, input_embeds)")
print("  load_weights(weights_iterator)")
print("\nNote: ForwardBatch replaces vLLM's separate kv_caches + attn_metadata")

## 4. The ForwardBatch Object

SGLang bundles all forward pass metadata into a `ForwardBatch` object.

In [ ]:
from dataclasses import dataclass

@dataclass
class SimulatedForwardBatch:
    """
    Simplified version of SGLang's ForwardBatch.
    
    Actual file: sglang/srt/layers/attention/forward_batch.py
    
    This object contains everything the model needs for a forward pass:
    - Input token information
    - Attention metadata (for FlashInfer)
    - KV cache references
    - Batch control information
    """
    
    # === Input information ===
    input_ids: torch.Tensor           # [num_tokens]
    seq_lens: torch.Tensor            # [batch_size] - length of each sequence
    positions: torch.Tensor           # [num_tokens] - position IDs
    
    # === Attention information ===
    forward_mode: str                 # "extend" (prefill) or "decode"
    out_cache_loc: torch.Tensor       # KV cache write locations
    
    # === Batch control ===
    batch_size: int                   # Number of sequences
    total_num_tokens: int             # Total tokens across all sequences
    
    # === FlashInfer specific ===
    # These are used by FlashInfer's paged KV cache
    req_pool_indices: Optional[torch.Tensor] = None
    seq_lens_sum: Optional[int] = None

# Demo
batch = SimulatedForwardBatch(
    input_ids=torch.tensor([100, 200, 300, 400, 500]),
    seq_lens=torch.tensor([3, 2]),  # Two sequences: 3 and 2 tokens
    positions=torch.tensor([0, 1, 2, 0, 1]),
    forward_mode="extend",  # Prefill mode
    out_cache_loc=torch.tensor([0, 1, 2, 3, 4]),
    batch_size=2,
    total_num_tokens=5,
)

print("ForwardBatch example:")
print(f"  Mode: {batch.forward_mode}")
print(f"  Batch size: {batch.batch_size}")
print(f"  Total tokens: {batch.total_num_tokens}")
print(f"  Sequence lengths: {batch.seq_lens.tolist()}")
print(f"  Input IDs: {batch.input_ids.tolist()}")
print(f"  Positions: {batch.positions.tolist()}")

## 5. SGLang Layer Primitives

SGLang shares many layer implementations with vLLM but has its own attention backend.

In [ ]:
# SGLang layer primitives

print("SGLang Layer Primitives")
print("=" * 60)

layers = {
    "RadixAttention": {
        "description": "SGLang's attention wrapper (equivalent to vLLM's Attention)",
        "features": [
            "Uses FlashInfer backend",
            "Supports extend (prefill) and decode modes",
            "Integrates with RadixCache for prefix caching",
            "Handles paged KV cache management",
        ],
        "import": "from sglang.srt.layers.attention import RadixAttention",
    },
    "Linear layers": {
        "description": "Reuses vLLM's parallel linear layers",
        "features": [
            "ColumnParallelLinear",
            "RowParallelLinear",
            "QKVParallelLinear",
            "MergedColumnParallelLinear",
        ],
        "import": "from vllm.model_executor.layers.linear import ...",
    },
    "Embedding layers": {
        "description": "Reuses vLLM's parallel embeddings",
        "features": [
            "VocabParallelEmbedding",
            "ParallelLMHead",
        ],
        "import": "from vllm.model_executor.layers.vocab_parallel_embedding import ...",
    },
    "Normalization": {
        "description": "Reuses vLLM's fused normalization kernels",
        "features": ["RMSNorm", "LayerNorm"],
        "import": "from vllm.model_executor.layers.layernorm import RMSNorm",
    },
    "Activation": {
        "description": "Fused activation functions",
        "features": ["SiluAndMul (fused SiLU + gate)"],
        "import": "from vllm.model_executor.layers.activation import SiluAndMul",
    },
}

for name, info in layers.items():
    print(f"\n{name}:")
    print(f"  {info['description']}")
    print(f"  Import: {info['import']}")
    print(f"  Features:")
    for feat in info['features']:
        print(f"    - {feat}")

## 6. RadixAttention: SGLang's Attention Wrapper

The key difference from vLLM is SGLang's `RadixAttention` class, which integrates with FlashInfer and RadixCache.

In [ ]:
# RadixAttention overview

RADIX_ATTENTION_CODE = '''
# File: sglang/srt/layers/attention/radix_attention.py (simplified)

class RadixAttention(nn.Module):
    """
    SGLang's attention layer.
    
    Key features:
    1. Uses FlashInfer for attention computation
    2. Supports paged KV cache
    3. Integrates with RadixCache for prefix sharing
    4. Handles extend (prefill) and decode modes
    """
    
    def __init__(
        self,
        num_heads: int,
        head_dim: int,
        scaling: float,
        num_kv_heads: int,
        layer_id: int,         # Layer index (for KV cache addressing)
        logit_cap: float = 0,  # For Gemma2-style logit capping
        v_head_dim: int = -1,  # For models with different V head dim
    ):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.scaling = scaling
        self.num_kv_heads = num_kv_heads
        self.layer_id = layer_id
    
    def forward(
        self,
        q: torch.Tensor,          # [num_tokens, num_heads, head_dim]
        k: torch.Tensor,          # [num_tokens, num_kv_heads, head_dim]
        v: torch.Tensor,          # [num_tokens, num_kv_heads, head_dim]
        forward_batch,            # ForwardBatch
    ) -> torch.Tensor:
        """
        Compute attention using FlashInfer.
        
        This method:
        1. Stores K, V into the paged KV cache
        2. Dispatches to FlashInfer for attention computation
        3. Handles both extend (prefill) and decode modes
        """
        # Store KV into cache
        forward_batch.token_to_kv_pool.set_kv_buffer(
            self.layer_id, forward_batch.out_cache_loc, k, v
        )
        
        # Dispatch to FlashInfer
        if forward_batch.forward_mode == "extend":
            # Prefill: process full sequence
            output = forward_batch.extend_attention(
                q, k, v, self.layer_id
            )
        else:
            # Decode: process one token per sequence
            output = forward_batch.decode_attention(
                q, self.layer_id
            )
        
        return output.view(-1, self.num_heads * self.head_dim)
'''

print(RADIX_ATTENTION_CODE)

print("\nKey differences from vLLM's Attention:")
print("1. Takes forward_batch instead of kv_cache + attn_metadata")
print("2. Uses FlashInfer backend instead of FlashAttention/xformers")
print("3. Has layer_id for addressing the correct KV cache layer")
print("4. Integrates with RadixCache for prefix sharing")

## 7. SGLang vs vLLM: Side-by-Side Comparison

In [ ]:
# Side-by-side comparison

comparison = [
    ("Registry location", 
     "vllm/model_executor/models/registry.py",
     "sglang/srt/models/registry.py"),
    ("Model package",
     "vllm.model_executor.models",
     "sglang.srt.models"),
    ("Attention wrapper",
     "vllm.attention.Attention",
     "sglang.srt.layers.attention.RadixAttention"),
    ("Attention backend",
     "FlashAttention2 / FlashInfer / xformers",
     "FlashInfer (primary)"),
    ("Forward signature",
     "forward(input_ids, positions, kv_caches, attn_metadata)",
     "forward(input_ids, positions, forward_batch)"),
    ("Linear layers",
     "vllm.model_executor.layers.linear",
     "Reuses vLLM's linear layers"),
    ("RoPE",
     "vllm.model_executor.layers.rotary_embedding",
     "Reuses vLLM's rotary embedding"),
    ("Weight loading",
     "load_weights(weights: Iterable[Tuple[str, Tensor]])",
     "load_weights(weights: Iterable[Tuple[str, Tensor]])"),
    ("User registration",
     "ModelRegistry.register_model(arch, cls)",
     "Similar API"),
    ("Quantization",
     "GPTQ, AWQ, FP8, Marlin, ...",
     "GPTQ, AWQ, FP8, Marlin (shared w/ vLLM)"),
]

print(f"{'Feature':<25} {'vLLM':<45} {'SGLang':<45}")
print("=" * 115)
for feature, vllm, sglang in comparison:
    print(f"{feature:<25} {vllm:<45} {sglang:<45}")

## 8. Weight Loading Pipeline in SGLang

In [ ]:
# SGLang weight loading pipeline

WEIGHT_LOADING_PIPELINE = '''
SGLang Weight Loading Pipeline
==============================

Step 1: Identify weight files
    - model.safetensors (single file)
    - model-00001-of-00003.safetensors (sharded)
    - pytorch_model.bin (legacy format)

Step 2: Create weight iterator
    - Streams weights one tensor at a time
    - Handles safetensors and PyTorch formats
    - Maps to correct device (GPU)

Step 3: Model's load_weights() method
    - Iterates over (name, tensor) pairs
    - Handles weight name mapping (HF -> SGLang)
    - Handles stacked parameters (QKV, gate_up)
    - Handles tensor parallelism sharding
    - Handles quantized weights

Step 4: Validation
    - Check all expected parameters are loaded
    - Warn about unexpected/missing weights
'''

print(WEIGHT_LOADING_PIPELINE)

In [ ]:
# SGLang weight loading implementation

def sglang_load_weights_example(
    model: nn.Module,
    weights,
):
    """
    Example load_weights implementation for SGLang.
    
    Essentially the same pattern as vLLM:
    1. Define stacked_params_mapping
    2. Iterate over weights
    3. Handle merging and direct copies
    """
    stacked_params_mapping = [
        # (sglang_merged_name, hf_shard_name, shard_id)
        ("qkv_proj", "q_proj", "q"),
        ("qkv_proj", "k_proj", "k"),
        ("qkv_proj", "v_proj", "v"),
        ("gate_up_proj", "gate_proj", 0),
        ("gate_up_proj", "up_proj", 1),
    ]
    
    params_dict = dict(model.named_parameters())
    loaded = set()
    
    for name, loaded_weight in weights:
        # Skip RoPE frequencies
        if "rotary_emb.inv_freq" in name:
            continue
        
        # Handle stacked params
        for (merged_name, shard_name, shard_id) in stacked_params_mapping:
            if shard_name not in name:
                continue
            name = name.replace(shard_name, merged_name)
            if name in params_dict:
                param = params_dict[name]
                weight_loader = getattr(param, "weight_loader", None)
                if weight_loader:
                    weight_loader(param, loaded_weight, shard_id)
                loaded.add(name)
            break
        else:
            # Direct copy
            if name in params_dict:
                param = params_dict[name]
                weight_loader = getattr(param, "weight_loader", None)
                if weight_loader:
                    weight_loader(param, loaded_weight)
                else:
                    param.data.copy_(loaded_weight)
                loaded.add(name)
    
    return loaded

print("SGLang weight loading follows the same pattern as vLLM:")
print("  1. stacked_params_mapping for merged weights")
print("  2. Iterate and handle each weight")
print("  3. Use weight_loader callbacks for TP-aware loading")

## 9. Configuration Mapping

In [ ]:
# Configuration mapping from HuggingFace to SGLang

print("Configuration Mapping: HuggingFace -> SGLang")
print("=" * 60)

config_mapping = [
    ("hidden_size", "Hidden dimension", "Direct mapping"),
    ("intermediate_size", "FFN intermediate dim", "Direct mapping"),
    ("num_attention_heads", "Number of Q heads", "Direct mapping"),
    ("num_key_value_heads", "Number of KV heads (GQA)", "Direct mapping"),
    ("num_hidden_layers", "Number of transformer layers", "Direct mapping"),
    ("vocab_size", "Vocabulary size", "Direct mapping"),
    ("max_position_embeddings", "Max sequence length", "Direct mapping"),
    ("rms_norm_eps", "RMSNorm epsilon", "Direct mapping"),
    ("rope_theta", "RoPE base frequency", "Direct mapping"),
    ("rope_scaling", "RoPE scaling config", "Parsed for YaRN/dynamic"),
    ("tie_word_embeddings", "Share embed/lm_head", "Handled in weight loading"),
    ("architectures", "Model architecture names", "Used for registry lookup"),
]

print(f"{'HF Config Field':<30} {'Description':<30} {'SGLang Handling':<25}")
print("-" * 85)
for field, desc, handling in config_mapping:
    print(f"{field:<30} {desc:<30} {handling:<25}")

print("\nSGLang reads the HuggingFace config directly via AutoConfig.from_pretrained()")
print("No separate config mapping is needed - the same config object is used.")

## 10. Source Code Walkthrough: SGLang Llama Model

In [ ]:
# SGLang's Llama implementation (annotated)

SGLANG_LLAMA_CODE = '''
# File: sglang/srt/models/llama.py (simplified)

import torch
import torch.nn as nn
from vllm.model_executor.layers.linear import (
    MergedColumnParallelLinear,
    QKVParallelLinear,
    RowParallelLinear,
)
from vllm.model_executor.layers.layernorm import RMSNorm
from vllm.model_executor.layers.rotary_embedding import get_rope
from vllm.model_executor.layers.vocab_parallel_embedding import (
    VocabParallelEmbedding, ParallelLMHead
)
from sglang.srt.layers.attention import RadixAttention  # <-- SGLang specific!
from sglang.srt.layers.logits_processor import LogitsProcessor


class LlamaMLP(nn.Module):
    def __init__(self, hidden_size, intermediate_size, hidden_act, quant_config):
        super().__init__()
        self.gate_up_proj = MergedColumnParallelLinear(
            hidden_size, [intermediate_size] * 2,
            bias=False, quant_config=quant_config,
        )
        self.down_proj = RowParallelLinear(
            intermediate_size, hidden_size,
            bias=False, quant_config=quant_config,
        )
        self.act_fn = SiluAndMul()
    
    def forward(self, x):
        gate_up, _ = self.gate_up_proj(x)
        x = self.act_fn(gate_up)
        x, _ = self.down_proj(x)
        return x


class LlamaAttention(nn.Module):
    def __init__(self, config, layer_id, quant_config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.head_dim = config.hidden_size // config.num_attention_heads
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        
        self.qkv_proj = QKVParallelLinear(
            config.hidden_size, self.head_dim,
            self.num_heads, self.num_kv_heads,
            bias=False, quant_config=quant_config,
        )
        self.o_proj = RowParallelLinear(
            config.hidden_size, config.hidden_size,
            bias=False, quant_config=quant_config,
        )
        self.rotary_emb = get_rope(
            self.head_dim,
            rotary_dim=self.head_dim,
            max_position=config.max_position_embeddings,
            base=config.rope_theta,
        )
        
        # KEY DIFFERENCE: RadixAttention instead of vLLM's Attention
        self.attn = RadixAttention(
            self.num_heads,
            self.head_dim,
            self.head_dim ** -0.5,
            self.num_kv_heads,
            layer_id=layer_id,  # <-- SGLang uses layer_id
        )
    
    def forward(self, hidden_states, positions, forward_batch):
        qkv, _ = self.qkv_proj(hidden_states)
        q, k, v = qkv.split(
            [self.num_heads * self.head_dim,
             self.num_kv_heads * self.head_dim,
             self.num_kv_heads * self.head_dim],
            dim=-1
        )
        q, k = self.rotary_emb(positions, q, k)
        
        # KEY DIFFERENCE: pass forward_batch instead of kv_cache + attn_metadata
        attn_output = self.attn(q, k, v, forward_batch)
        
        output, _ = self.o_proj(attn_output)
        return output


class LlamaForCausalLM(nn.Module):
    def __init__(self, config, quant_config=None):
        super().__init__()
        self.config = config
        self.model = LlamaModel(config, quant_config)
        self.lm_head = ParallelLMHead(config.vocab_size, config.hidden_size)
        self.logits_processor = LogitsProcessor(config)
    
    def forward(self, input_ids, positions, forward_batch, input_embeds=None):
        hidden_states = self.model(input_ids, positions, forward_batch, input_embeds)
        return self.logits_processor(
            input_ids, hidden_states, self.lm_head.weight, forward_batch
        )
    
    def load_weights(self, weights):
        # Same pattern as vLLM (omitted for brevity)
        ...
'''

print(SGLANG_LLAMA_CODE)

## 11. Registering a Custom Model in SGLang

In [ ]:
# Step-by-step: Register a custom model in SGLang

REGISTRATION_STEPS = '''
Registering a Custom Model in SGLang
=====================================

Method 1: Add to the registry (for contributing to SGLang)
---------------------------------------------------------

Step 1: Create your model file
    sglang/srt/models/my_model.py

Step 2: Add to the registry
    # In sglang/srt/models/registry.py
    _MODELS = {
        ...
        "MyModelForCausalLM": ("my_model", "MyModelForCausalLM"),
    }

Method 2: External registration (for users)
--------------------------------------------

Step 1: Create your model class
    # my_model.py
    class MyModelForCausalLM(nn.Module):
        ...

Step 2: Register before launching the server
    from sglang.srt.models.registry import ModelRegistry
    from my_model import MyModelForCausalLM
    
    ModelRegistry.register("MyModelForCausalLM", MyModelForCausalLM)
    
    # Then launch
    import sglang as sgl
    sgl.launch_server(model_path="path/to/my_model", ...)
'''

print(REGISTRATION_STEPS)

In [ ]:
# Complete example: A toy model for SGLang

class ToyForCausalLM_SGLang(nn.Module):
    """
    A toy model implementing SGLang's model interface.
    """
    
    def __init__(self, config, quant_config=None):
        super().__init__()
        self.config = config
        
        # Embedding
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        
        # Simple transformer layers
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=config.hidden_size,
                nhead=config.num_attention_heads,
                dim_feedforward=config.intermediate_size,
                batch_first=True,
            )
            for _ in range(config.num_hidden_layers)
        ])
        
        self.norm = nn.LayerNorm(config.hidden_size)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
    
    def forward(
        self,
        input_ids: torch.Tensor,
        positions: torch.Tensor,
        forward_batch,  # ForwardBatch in real SGLang
        input_embeds: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """SGLang forward pass."""
        if input_embeds is not None:
            hidden_states = input_embeds
        else:
            hidden_states = self.embed_tokens(input_ids)
        
        # Add batch dimension if needed
        if hidden_states.dim() == 2:
            hidden_states = hidden_states.unsqueeze(0)
        
        for layer in self.layers:
            hidden_states = layer(hidden_states)
        
        hidden_states = self.norm(hidden_states)
        logits = self.lm_head(hidden_states)
        
        return logits.squeeze(0)  # Remove batch dim
    
    def load_weights(self, weights):
        """Load weights from HuggingFace checkpoint."""
        params_dict = dict(self.named_parameters())
        loaded = set()
        
        for name, tensor in weights:
            if name in params_dict:
                params_dict[name].data.copy_(tensor)
                loaded.add(name)
        
        return loaded

# Test
class ToyConfig:
    vocab_size = 1000
    hidden_size = 256
    num_attention_heads = 4
    num_hidden_layers = 2
    intermediate_size = 512

model = ToyForCausalLM_SGLang(ToyConfig())
input_ids = torch.randint(0, 1000, (16,))
positions = torch.arange(16)

with torch.no_grad():
    logits = model(input_ids, positions, forward_batch=None)

print(f"SGLang Toy Model:")
print(f"  Input: {input_ids.shape}")
print(f"  Output: {logits.shape}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 12. SGLang's LogitsProcessor

SGLang handles logits processing differently from vLLM.

In [ ]:
# SGLang LogitsProcessor

LOGITS_PROCESSOR_CODE = '''
# File: sglang/srt/layers/logits_processor.py (simplified)

class LogitsProcessor(nn.Module):
    """
    SGLang's logits processor.
    
    Key features:
    1. Computes logits from hidden states
    2. Handles vocabulary parallelism
    3. Only computes logits for the LAST token of each sequence
       (optimization: we only need logits for sampling)
    """
    
    def __init__(self, config):
        super().__init__()
        self.config = config
    
    def forward(
        self,
        input_ids,        # [num_tokens]
        hidden_states,    # [num_tokens, hidden_size]
        weight,           # lm_head weight [vocab_size, hidden_size]
        forward_batch,    # ForwardBatch
    ):
        """
        Compute logits.
        
        Optimization: Only compute logits for the last token
        of each sequence in the batch.
        
        During prefill: only the last token needs logits
        During decode: every token is the last token
        """
        if forward_batch.forward_mode == "extend":
            # Get indices of last token in each sequence
            last_indices = torch.cumsum(forward_batch.seq_lens, dim=0) - 1
            hidden_states = hidden_states[last_indices]
        
        # Compute logits
        logits = torch.matmul(hidden_states, weight.t())
        
        return logits  # [batch_size, vocab_size]
'''

print(LOGITS_PROCESSOR_CODE)
print("\nKey optimization: During prefill, only compute logits for")
print("the last token of each sequence (saves compute and memory).")

## 13. Exercises

### Exercise 1: Compare Registration
Given a HuggingFace model with architecture name "NewModelForCausalLM", write the registration code for both vLLM and SGLang.

### Exercise 2: Adapt a vLLM Model for SGLang
Take the Qwen2-style model from Chapter 8.2 and adapt it for SGLang. The main change is replacing the Attention wrapper and forward() signature.

### Exercise 3: LogitsProcessor Optimization
Implement the optimization where logits are only computed for the last token of each sequence during prefill. Measure the speedup.

In [ ]:
# Exercise 1: Side-by-side registration

print("Exercise 1: Register 'NewModelForCausalLM'")
print("\nvLLM:")
print("""  # In vllm/model_executor/models/registry.py
  _GENERATION_MODELS = {
      ...
      "NewModelForCausalLM": ("new_model", "NewModelForCausalLM"),
  }
  
  # Or externally:
  from vllm import ModelRegistry
  ModelRegistry.register_model("NewModelForCausalLM", NewModelForCausalLM)
""")

print("SGLang:")
print("""  # In sglang/srt/models/registry.py
  _MODELS = {
      ...
      "NewModelForCausalLM": ("new_model", "NewModelForCausalLM"),
  }
  
  # Or externally:
  from sglang.srt.models.registry import ModelRegistry
  ModelRegistry.register("NewModelForCausalLM", NewModelForCausalLM)
""")

In [ ]:
# Exercise 3: LogitsProcessor optimization
import time

def logits_all_tokens(hidden_states, weight):
    """Compute logits for ALL tokens."""
    return hidden_states @ weight.t()

def logits_last_only(hidden_states, weight, seq_lens):
    """Compute logits for LAST tokens only."""
    last_indices = torch.cumsum(seq_lens, dim=0) - 1
    last_hidden = hidden_states[last_indices]
    return last_hidden @ weight.t()

# Benchmark
num_tokens = 2048
batch_size = 8
hidden_size = 4096
vocab_size = 32000

hidden_states = torch.randn(num_tokens, hidden_size)
weight = torch.randn(vocab_size, hidden_size)
seq_lens = torch.full((batch_size,), num_tokens // batch_size)

# Time both approaches
start = time.time()
for _ in range(10):
    all_logits = logits_all_tokens(hidden_states, weight)
t_all = (time.time() - start) / 10

start = time.time()
for _ in range(10):
    last_logits = logits_last_only(hidden_states, weight, seq_lens)
t_last = (time.time() - start) / 10

print(f"All tokens logits:  shape={all_logits.shape}, time={t_all*1000:.1f}ms")
print(f"Last token logits:  shape={last_logits.shape}, time={t_last*1000:.1f}ms")
print(f"Speedup: {t_all/t_last:.1f}x")
print(f"Memory saved: {(num_tokens - batch_size) * vocab_size * 4 / 1024**2:.0f} MB")

## Summary

In this notebook, we covered:

1. **SGLang Registry**: Maps architecture names to model classes (similar to vLLM)
2. **Model Interface**: `forward(input_ids, positions, forward_batch)` with ForwardBatch
3. **RadixAttention**: SGLang's attention wrapper using FlashInfer
4. **Shared Layers**: SGLang reuses vLLM's linear layers, norms, and embeddings
5. **Weight Loading**: Same stacked_params_mapping pattern as vLLM
6. **LogitsProcessor**: Optimized to only compute last-token logits during prefill

### Key Takeaway
SGLang and vLLM share most model implementation code (linear layers, norms, RoPE). The main differences are the attention backend (RadixAttention vs Attention) and the forward() signature (ForwardBatch vs kv_caches + attn_metadata).

### Next: Chapter 8.6 -- Implementing a New Model in SGLang